In [1]:
import os
os.chdir("../")
%pwd

'/home/muggle/hlt/py/ds/cmpmlops/datascienceproject'

In [2]:
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/Muggle/datascienceproject.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "Muggle"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "4c86eb70ba6719804ff9622fb4007b22bee2a55c"

In [3]:
from dataclasses import dataclass
from pathlib import Path

In [4]:
@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [5]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [6]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN
        create_directories([config.root_dir])
        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema.name,
            mlflow_uri="https://dagshub.com/Muggle/datascienceproject.mlflow"
        )
        return model_evaluation_config

In [7]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [10]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]
        
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_artifact_uri()).scheme

        if mlflow.active_run():
            print("Ending the active run...")
            mlflow.end_run()

        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

            score = {
                "rmse": rmse,
                "mae": mae,
                "r2": r2
            }
            save_json(path=Path(self.config.metric_file_name), data=score)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, "model")

In [11]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e
    

[2026-01-01 02:25:15,558: INFO: common: yaml file: config/config.yaml loaded successfilly]
[2026-01-01 02:25:15,559: INFO: common: yaml file: params.yaml loaded successfilly]
[2026-01-01 02:25:15,560: INFO: common: yaml file: schema.yaml loaded successfilly]
[2026-01-01 02:25:15,560: INFO: common: created directory at: artifacts]
[2026-01-01 02:25:15,561: INFO: common: created directory at: artifacts/model_evaluation]
Ending the active run...
🏃 View run omniscient-perch-147 at: https://dagshub.com/Muggle/datascienceproject.mlflow/#/experiments/0/runs/14747fc6f5144f2295114843b6490012
🧪 View experiment at: https://dagshub.com/Muggle/datascienceproject.mlflow/#/experiments/0
[2026-01-01 02:25:17,495: INFO: common: json file saved at: artifacts/model_evaluation/metrics.json]


2026/01/01 02:25:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'ElasticnetModel'.
2026/01/01 02:25:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 1
Created version '1' of model 'ElasticnetModel'.


🏃 View run defiant-carp-274 at: https://dagshub.com/Muggle/datascienceproject.mlflow/#/experiments/0/runs/48f9f78ab2c448dab51f703150baac70
🧪 View experiment at: https://dagshub.com/Muggle/datascienceproject.mlflow/#/experiments/0
